# Notebook 04 — Đánh giá mô hình (Evaluation)
**DSP501 — Nhận diện người nói (Speaker Identification)**

## Mục tiêu
So sánh 2 pipeline xử lý tín hiệu âm thanh:

| Pipeline | Mô tả | Features |
|----------|-------|----------|
| **A** (Không có DSP) | Audio thô → RMS, ZCR, amplitude | 6 chiều |
| **B** (Có DSP) | Audio → FIR filter → Pre-emphasis → MFCC | 26 chiều |

## Phương pháp đánh giá
- **Train/Test split**: 80% huấn luyện, 20% kiểm tra (stratified — giữ tỷ lệ speakers đều)
- **Classifier**: SVM với RBF kernel + GridSearchCV tìm hyperparameters tốt nhất
- **Metrics**: Accuracy, Precision, Recall, F1-score, Confusion Matrix, ROC Curve
- **Statistical test**: Paired t-test kiểm tra sự khác biệt có ý nghĩa thống kê hay không

In [ ]:
import sys
sys.path.insert(0, '../src')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler, label_binarize
from sklearn.pipeline import Pipeline
from sklearn.model_selection import (
    train_test_split, StratifiedKFold, RepeatedStratifiedKFold,
    GridSearchCV, cross_val_score
)
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score,
    confusion_matrix, classification_report,
    roc_curve, auc
)
from evaluation import compute_ci, paired_ttest

%matplotlib inline
plt.rcParams['figure.dpi'] = 120
plt.rcParams['figure.facecolor'] = 'white'

RANDOM_SEED = 42
print('Import thành công!')

---
## 1. Tải dữ liệu đặc trưng (Load Features)

Dữ liệu đã được trích xuất trước bởi `feature_extraction.py` và lưu dưới dạng `.npy`:

- `features_basic.npy` — Pipeline A: 6 đặc trưng cơ bản (RMS mean/std, ZCR mean/std, mean|amplitude|, std amplitude)
- `features_mfcc_filt.npy` — Pipeline B: 26 đặc trưng MFCC (13 mean + 13 std) sau khi qua FIR filter + pre-emphasis
- `labels.npy` — nhãn speaker_id cho từng mẫu

In [ ]:
# Tải features đã trích xuất sẵn
X_basic = np.load('../features/features_basic.npy')    # Pipeline A: 6 chiều
X_mfcc  = np.load('../features/features_mfcc_filt.npy') # Pipeline B: 26 chiều
y       = np.load('../features/labels.npy')              # Nhãn speaker

# Ánh xạ speaker_id -> tên (speaker_id = folder number)
speaker_names = {7: 'Dan', 8: 'Cuong', 9: 'Quang', 10: 'Anne', 11: 'Khoa'}
labels_sorted = sorted(np.unique(y))
label_names = [speaker_names.get(i, f'Spk {i}') for i in labels_sorted]

print(f'=== THÔNG TIN DATASET ===')
print(f'Pipeline A: {X_basic.shape[0]} mẫu x {X_basic.shape[1]} đặc trưng (basic time-domain)')
print(f'Pipeline B: {X_mfcc.shape[0]} mẫu x {X_mfcc.shape[1]} đặc trưng (MFCC sau DSP)')
print(f'Số speakers: {len(labels_sorted)}')
print(f'Tên speakers: {label_names}')
print(f'Số mẫu mỗi speaker: {dict(zip(label_names, [np.sum(y == i) for i in labels_sorted]))}')
print(f'\nVí dụ 1 mẫu Pipeline A (6 đặc trưng):')
print(f'  {X_basic[0]}')
print(f'\nVí dụ 1 mẫu Pipeline B (26 đặc trưng, hiển thị 10 đầu):')
print(f'  {X_mfcc[0][:10]} ...')

---
## 2. Chia tập Train/Test (Block Split — 70/30)

### Vấn đề với Random Split khi dataset nhỏ

Các mẫu audio cùng speaker được thu trong **cùng 1 session** → MFCC rất giống nhau (correlation > 0.98).
Nếu dùng random split, test samples gần như **bản sao** của train samples → accuracy = 1.0 nhưng **không thể hiện khả năng thực sự**.

### Block Split là gì?

Thay vì chia ngẫu nhiên, ta chia **theo thứ tự file**:
- Mỗi speaker có 25 file (01.wav → 25.wav)
- **70% đầu** (file 01-17) → train
- **30% cuối** (file 18-25) → test

Điều này đảm bảo test set chứa audio **khác thời điểm** so với train set → đánh giá chính xác hơn.

In [ ]:
# Block split: mỗi speaker, 70% file đầu → train, 30% file cuối → test
# Tránh data leakage do các mẫu cùng session quá giống nhau

train_idx, test_idx = [], []
for spk in labels_sorted:
    spk_indices = np.where(y == spk)[0]
    n_train = int(len(spk_indices) * 0.7)
    train_idx.extend(spk_indices[:n_train])
    test_idx.extend(spk_indices[n_train:])

train_idx = np.array(train_idx)
test_idx = np.array(test_idx)

X_basic_train, X_basic_test = X_basic[train_idx], X_basic[test_idx]
X_mfcc_train, X_mfcc_test   = X_mfcc[train_idx], X_mfcc[test_idx]
y_train, y_test             = y[train_idx], y[test_idx]

print('=== CHIA DỮ LIỆU (Block Split) ===')
print(f'Tổng: {len(y)} mẫu')
print(f'Train: {len(y_train)} mẫu (70% file đầu mỗi speaker)')
print(f'Test:  {len(y_test)} mẫu (30% file cuối mỗi speaker)')
print(f'\nPhân bố train:')
for name, sid in zip(label_names, labels_sorted):
    print(f'  {name}: {np.sum(y_train == sid)} mẫu')
print(f'\nPhân bố test:')
for name, sid in zip(label_names, labels_sorted):
    print(f'  {name}: {np.sum(y_test == sid)} mẫu')
print(f'\nTại sao block split? Correlation giữa mẫu cùng speaker > 0.98')
print(f'→ Random split sẽ "leak" thông tin → accuracy giả cao')

---
## 3. Huấn luyện SVM (Train Models)

### SVM (Support Vector Machine) là gì?
- Thuật toán phân loại tìm **ranh giới tối ưu** (hyperplane) giữa các nhóm speakers
- Kernel **RBF** (Radial Basis Function): cho phép ranh giới phi tuyến (cong), phù hợp với dữ liệu phức tạp

### GridSearchCV là gì?
- Thử tất cả tổ hợp hyperparameters (C, gamma) để tìm bộ tốt nhất
- **C**: kiểm soát lỗi (C lớn → ít lỗi trên train, có thể overfit)
- **gamma**: kiểm soát phạm vi ảnh hưởng (gamma lớn → mỗi mẫu ảnh hưởng gần hơn)

### RepeatedStratifiedKFold là gì?
- Lặp lại cross-validation nhiều lần (5-fold × 10 lần = 50 evaluations)
- Giảm phương sai do chia fold ngẫu nhiên → kết quả **ổn định** và **đáng tin cậy hơn**
- Đặc biệt quan trọng khi dataset nhỏ (100 mẫu)

In [ ]:
# Các hyperparameters cần thử — giảm C max để tránh overfit
SVM_PARAM_GRID = {
    'svm__C':     [0.01, 0.1, 1, 10],              # C nhỏ hơn → regularize mạnh hơn
    'svm__gamma': ['scale', 'auto', 0.001, 0.01],   # 4 giá trị
}   # Tổng: 4 x 4 = 16 tổ hợp sẽ được thử

# RepeatedStratifiedKFold: 5-fold × 10 repeats = 50 evaluations → ổn định hơn
RSKF = RepeatedStratifiedKFold(n_splits=5, n_repeats=10, random_state=RANDOM_SEED)

def train_and_evaluate(X_train, X_test, y_train, y_test, name):
    """Huấn luyện SVM, tìm params tốt nhất, đánh giá trên test set."""
    # Bước 1: Tạo pipeline (StandardScaler chuẩn hóa + SVM)
    pipe = Pipeline([
        ('scaler', StandardScaler()),  # Chuẩn hóa features về mean=0, std=1
        ('svm', SVC(kernel='rbf', probability=True, random_state=RANDOM_SEED)),
    ])
    
    # Bước 2: GridSearchCV tìm hyperparameters tốt nhất (dùng 5-fold CV)
    grid = GridSearchCV(pipe, SVM_PARAM_GRID, cv=5, scoring='accuracy', n_jobs=-1, refit=True)
    grid.fit(X_train, y_train)
    
    best_model = grid.best_estimator_
    best_params = {k.replace('svm__', ''): v for k, v in grid.best_params_.items()}
    
    # Bước 3: Dự đoán trên test set (dữ liệu MỚI, chưa thấy bao giờ)
    y_pred = best_model.predict(X_test)
    y_prob = best_model.predict_proba(X_test)  # Xác suất cho ROC curve
    
    # Bước 4: RepeatedStratifiedKFold trên train set (50 evaluations → đáng tin cậy)
    cv_scores = cross_val_score(best_model, X_train, y_train, cv=RSKF, scoring='accuracy')
    
    # In kết quả
    print(f"\n{'='*55}")
    print(f'  {name}')
    print(f"{'='*55}")
    print(f'  Hyperparameters tốt nhất: {best_params}')
    print(f'  CV Accuracy (5-fold × 10 repeats): {cv_scores.mean():.4f} +/- {cv_scores.std():.4f}')
    print(f'  Test Accuracy: {accuracy_score(y_test, y_pred):.4f}')
    print(f'  Gap (test - CV): {accuracy_score(y_test, y_pred) - cv_scores.mean():+.4f}')
    
    return best_model, y_pred, y_prob, cv_scores

# --- Huấn luyện Pipeline A (không có DSP) ---
model_a, y_pred_a, y_prob_a, cv_a = train_and_evaluate(
    X_basic_train, X_basic_test, y_train, y_test,
    'Pipeline A — Không có DSP (basic features)')

# --- Huấn luyện Pipeline B (có DSP) ---
model_b, y_pred_b, y_prob_b, cv_b = train_and_evaluate(
    X_mfcc_train, X_mfcc_test, y_train, y_test,
    'Pipeline B — Có DSP (FIR + Pre-emphasis + MFCC)')

---
## 4. Bảng chỉ số đánh giá (Metrics Table)

### Giải thích các chỉ số:

| Chỉ số | Ý nghĩa | Ví dụ cụ thể |
|--------|---------|---------------|
| **Accuracy** | Tỷ lệ dự đoán đúng trên tổng số mẫu | 18/20 đúng = 90% |
| **Precision** | Khi model nói "đây là Cuong", bao nhiêu % thực sự là Cuong? | 5 lần nói Cuong, 4 lần đúng = 80% |
| **Recall** | Trong tất cả mẫu thật của Cuong, model tìm được bao nhiêu %? | 5 mẫu Cuong, model tìm đúng 4 = 80% |
| **F1-score** | Trung bình hài hòa của Precision và Recall | Cân bằng giữa 2 chỉ số trên |
| **CV Mean** | Accuracy trung bình qua 5 lần cross-validation | Đánh giá độ ổn định |
| **95% CI** | Khoảng tin cậy 95% | "Accuracy thực sự nằm trong khoảng [0.70, 0.98]" |

In [ ]:
# Tính tất cả metrics trên TEST set
def get_metrics(y_true, y_pred, cv_scores):
    ci = compute_ci(cv_scores)
    return {
        'Accuracy': accuracy_score(y_true, y_pred),
        'Precision': precision_score(y_true, y_pred, average='macro', zero_division=0),
        'Recall': recall_score(y_true, y_pred, average='macro', zero_division=0),
        'F1-score': f1_score(y_true, y_pred, average='macro', zero_division=0),
        'CV Mean +/- Std': f'{cv_scores.mean():.3f} +/- {cv_scores.std():.3f}',
        '95% CI': f'[{ci[0]:.3f}, {ci[1]:.3f}]',
    }

metrics_a = get_metrics(y_test, y_pred_a, cv_a)
metrics_b = get_metrics(y_test, y_pred_b, cv_b)

# Hiển thị bảng so sánh
df_metrics = pd.DataFrame(
    [metrics_a, metrics_b],
    index=['Pipeline A (Không DSP)', 'Pipeline B (Có DSP)']
).T

print('=' * 60)
print('       BẢNG ĐÁNH GIÁ TRÊN TEST SET')
print('=' * 60)
df_metrics

### Báo cáo chi tiết từng speaker

Classification report cho thấy Precision, Recall, F1 **cho từng speaker riêng biệt**.

Ví dụ: nếu Cuong có Recall = 0.60, nghĩa là model chỉ nhận đúng 60% mẫu của Cuong → model yếu với giọng Cuong.

In [ ]:
print('PIPELINE A — Báo cáo chi tiết từng speaker (Test Set)')
print(classification_report(y_test, y_pred_a, target_names=label_names))

print('\nPIPELINE B — Báo cáo chi tiết từng speaker (Test Set)')
print(classification_report(y_test, y_pred_b, target_names=label_names))

---
## 5. Ma trận nhầm lẫn (Confusion Matrix)

### Đọc Confusion Matrix như thế nào?

- **Hàng** = speaker thật (Actual)
- **Cột** = speaker model dự đoán (Predicted)
- **Đường chéo** = dự đoán ĐÚNG (số càng lớn càng tốt)
- **Ngoài đường chéo** = dự đoán SAI (nhầm speaker này thành speaker khác)

**Ví dụ:** Nếu ô (Cuong, Quang) = 2 → model nhầm 2 mẫu của Cuong thành Quang

In [ ]:
import os
os.makedirs('../figures', exist_ok=True)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Pipeline A — Confusion Matrix
cm_a = confusion_matrix(y_test, y_pred_a)
sns.heatmap(cm_a, annot=True, fmt='d', cmap='Blues',
            xticklabels=label_names, yticklabels=label_names, ax=axes[0])
axes[0].set_title('Pipeline A — Không có DSP')
axes[0].set_xlabel('Dự đoán (Predicted)')
axes[0].set_ylabel('Thực tế (Actual)')

# Pipeline B — Confusion Matrix
cm_b = confusion_matrix(y_test, y_pred_b)
sns.heatmap(cm_b, annot=True, fmt='d', cmap='Oranges',
            xticklabels=label_names, yticklabels=label_names, ax=axes[1])
axes[1].set_title('Pipeline B — Có DSP')
axes[1].set_xlabel('Dự đoán (Predicted)')
axes[1].set_ylabel('Thực tế (Actual)')

plt.suptitle('Ma trận nhầm lẫn (Confusion Matrix) — Test Set', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../figures/confusion_matrices.png', dpi=150, bbox_inches='tight')
plt.show()

print('\nĐọc kết quả:')
print(f'  Pipeline A: {np.trace(cm_a)}/{cm_a.sum()} dự đoán đúng trên test set')
print(f'  Pipeline B: {np.trace(cm_b)}/{cm_b.sum()} dự đoán đúng trên test set')

---
## 6. Đường cong ROC (ROC Curve)

### ROC Curve là gì?

- Đo khả năng **phân biệt** giữa các speakers
- Trục X: **False Positive Rate** (tỷ lệ nhầm — model nói đúng nhưng sai)
- Trục Y: **True Positive Rate** (tỷ lệ đúng — model nói đúng và đúng thật)
- **AUC** (Area Under Curve): diện tích dưới đường cong, từ 0 đến 1

### Cách đọc:
- Đường **càng gần góc trái trên** (0, 1) → model càng tốt
- **AUC = 1.00** → phân biệt hoàn hảo
- **AUC = 0.50** → đoán ngẫu nhiên (đường chéo nét đứt)
- **AUC > 0.90** → rất tốt

Vì có 4 speakers (multi-class), ta dùng phương pháp **One-vs-Rest**: mỗi speaker được so sánh với tất cả speakers còn lại.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd']

# Tạo one-hot encoding thủ công (tránh lỗi label_binarize với IDs không liên tục)
classes = labels_sorted  # [7, 8, 9, 10]
n_classes = len(classes)
y_test_bin = np.zeros((len(y_test), n_classes))
for i, cls in enumerate(classes):
    y_test_bin[:, i] = (y_test == cls).astype(int)

for ax, model, y_prob, title in [
    (axes[0], model_a, y_prob_a, 'Pipeline A — Không có DSP'),
    (axes[1], model_b, y_prob_b, 'Pipeline B — Có DSP'),
]:
    # Đảm bảo thứ tự cột y_prob khớp với classes
    model_classes = list(model.classes_)
    
    for i, cls in enumerate(classes):
        # Tìm cột tương ứng trong y_prob (theo model.classes_)
        prob_idx = model_classes.index(cls)
        fpr, tpr, _ = roc_curve(y_test_bin[:, i], y_prob[:, prob_idx])
        roc_auc = auc(fpr, tpr)
        ax.plot(fpr, tpr, color=colors[i % len(colors)],
                label=f'{label_names[i]} (AUC={roc_auc:.2f})', linewidth=2)
    
    # Đường chéo = đoán ngẫu nhiên
    ax.plot([0, 1], [0, 1], 'k--', linewidth=0.8, alpha=0.5, label='Ngẫu nhiên (AUC=0.50)')
    ax.set_title(title)
    ax.set_xlabel('False Positive Rate (Tỷ lệ nhầm)')
    ax.set_ylabel('True Positive Rate (Tỷ lệ đúng)')
    ax.legend(loc='lower right', fontsize=8)
    ax.grid(True, alpha=0.3)
    ax.set_xlim([-0.02, 1.02])
    ax.set_ylim([-0.02, 1.02])

plt.suptitle('Đường cong ROC — One-vs-Rest (Test Set)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../figures/roc_curves.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 7. So sánh 2 Pipeline — Biểu đồ cột

Biểu đồ dưới đây so sánh trực quan **4 chỉ số** giữa Pipeline A và Pipeline B trên **test set**.

In [ ]:
metrics_names = ['Accuracy', 'Precision', 'Recall', 'F1-score']
scores_a = [metrics_a[m] for m in metrics_names]
scores_b = [metrics_b[m] for m in metrics_names]

x = np.arange(len(metrics_names))
width = 0.35

fig, ax = plt.subplots(figsize=(9, 5))
bars_a = ax.bar(x - width/2, scores_a, width,
                label='Pipeline A (Không DSP)', color='steelblue', alpha=0.85)
bars_b = ax.bar(x + width/2, scores_b, width,
                label='Pipeline B (Có DSP)', color='darkorange', alpha=0.85)

# Hiển thị giá trị trên mỗi cột
for bars in [bars_a, bars_b]:
    for bar in bars:
        h = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2, h + 0.01,
                f'{h:.3f}', ha='center', va='bottom', fontsize=10)

ax.set_ylim(0, 1.15)
ax.set_ylabel('Điểm số')
ax.set_title('So sánh Pipeline A vs Pipeline B — Tất cả chỉ số (Test Set)',
             fontsize=13, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(metrics_names)
ax.legend(loc='upper left')
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.savefig('../figures/comparison_bar.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 8. Kiểm định thống kê — Paired t-test

### Tại sao cần kiểm định thống kê?

Pipeline B có accuracy cao hơn Pipeline A, nhưng liệu sự khác biệt này có **thật sự ý nghĩa** hay chỉ do **may mắn**?

### Paired t-test hoạt động thế nào?

1. Lấy 5 CV scores của Pipeline A và 5 CV scores của Pipeline B
2. Tính hiệu số từng cặp: d₁ = B₁ - A₁, d₂ = B₂ - A₂, ...
3. Kiểm tra: hiệu số trung bình có khác 0 không?

### Cách đọc kết quả:
- **p-value < 0.05** → Sự khác biệt có ý nghĩa thống kê (confident > 95%)
- **p-value >= 0.05** → Không đủ bằng chứng để kết luận

**Ví dụ:** p = 0.035 → chỉ có 3.5% khả năng kết quả này xảy ra do ngẫu nhiên → **có ý nghĩa thống kê**

In [ ]:
t_stat, p_value = paired_ttest(cv_a, cv_b)

print('=' * 60)
print('   KIỂM ĐỊNH THỐNG KÊ: Paired t-test (A vs B)')
print('=' * 60)
print(f'  t-statistic : {t_stat:.4f}')
print(f'  p-value     : {p_value:.4f}')
print()
print(f'  Pipeline A — 5 fold scores: {np.round(cv_a, 3)}')
print(f'  Pipeline A — trung bình:    {cv_a.mean():.4f}')
print()
print(f'  Pipeline B — 5 fold scores: {np.round(cv_b, 3)}')
print(f'  Pipeline B — trung bình:    {cv_b.mean():.4f}')
print()
print(f'  Chênh lệch trung bình: {cv_b.mean() - cv_a.mean():+.4f}')
print()
if p_value < 0.05:
    print(f'  >>> KẾT LUẬN: DSP preprocessing CẢI THIỆN CÓ Ý NGHĨA thống kê (p={p_value:.4f} < 0.05)')
else:
    print(f'  >>> KẾT LUẬN: Chưa đủ bằng chứng để kết luận (p={p_value:.4f} >= 0.05)')

---
## 9. Tổng kết (Summary)

In [ ]:
print('=' * 65)
print('                    TỔNG KẾT ĐÁNH GIÁ')
print('=' * 65)
print(f"  Dataset: {len(y)} mẫu, {len(labels_sorted)} speakers ({', '.join(label_names)})")
print(f'  Chia dữ liệu: {len(y_train)} train / {len(y_test)} test (block split 70/30)')
print(f'  Cross-validation: RepeatedStratifiedKFold (5-fold × 10 repeats)')
print()
print(f'  ┌─────────────────────────────────────────────────────────┐')
print(f'  │  Pipeline A (Không DSP):                                │')
print(f'  │    Features:      RMS, ZCR, amplitude (6 chiều)         │')
print(f'  │    Test Accuracy:  {metrics_a["Accuracy"]:.4f}                              │')
print(f'  │    CV Accuracy:    {cv_a.mean():.4f} +/- {cv_a.std():.4f}                   │')
print(f'  │    Test F1-score:  {metrics_a["F1-score"]:.4f}                              │')
print(f'  ├─────────────────────────────────────────────────────────┤')
print(f'  │  Pipeline B (Có DSP):                                   │')
print(f'  │    Features:      FIR + Pre-emphasis + MFCC (26 chiều)  │')
print(f'  │    Test Accuracy:  {metrics_b["Accuracy"]:.4f}                              │')
print(f'  │    CV Accuracy:    {cv_b.mean():.4f} +/- {cv_b.std():.4f}                   │')
print(f'  │    Test F1-score:  {metrics_b["F1-score"]:.4f}                              │')
print(f'  ├─────────────────────────────────────────────────────────┤')
print(f'  │  Cải thiện:  +{metrics_b["Accuracy"] - metrics_a["Accuracy"]:.4f} accuracy (test)                │')
print(f'  │              +{cv_b.mean() - cv_a.mean():.4f} accuracy (CV)                  │')
print(f'  │  p-value:    {p_value:.4f} ({"CÓ" if p_value < 0.05 else "KHÔNG"} ý nghĩa thống kê)             │')
print(f'  └─────────────────────────────────────────────────────────┘')
print()
print('  Phương pháp: Block split (file đầu → train, file cuối → test)')
print('  tránh data leakage do mẫu cùng session quá giống nhau.')
print()
print('  Kết luận: DSP preprocessing (FIR filter + Pre-emphasis + MFCC)')
print('  giúp cải thiện đáng kể khả năng nhận diện người nói so với')
print('  chỉ dùng đặc trưng cơ bản (RMS, ZCR).')